In [ ]:
# CPU allocation controls. Set CPU_LIST like "0-15" before executing if desired.
import os

CPU_LIST = None

def parse_cpu_list(spec):
    if spec is None or str(spec).strip() == "":
        return None
    cpus = []
    for raw_part in str(spec).split(","):
        part = raw_part.strip()
        if not part:
            continue
        if "-" in part:
            start, stop = (int(x) for x in part.split("-", 1))
            cpus.extend(range(start, stop + 1))
        else:
            cpus.append(int(part))
    return sorted(set(cpus))

requested_cpus = parse_cpu_list(CPU_LIST)
if requested_cpus is not None and hasattr(os, "sched_setaffinity"):
    os.sched_setaffinity(0, requested_cpus)
effective_cpus = sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
print(f"CPU_LIST requested={CPU_LIST!r}; effective={effective_cpus}")


In [ ]:
from pathlib import Path
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "REPO_POLICY.md").exists():
    REPO_ROOT = Path.cwd().parent

CAMPAIGN_ROOT = (
    REPO_ROOT / "choi_covariance_cpu" / "cpu_data" / "complex_particle_choi_transfer"
    / "campaigns" / "Nx16_Ny16-24-32_DW1_dwtrunc1_alpha1_sweep_S10_cycles2Ny_v5"
)
FIGURE_DIR = REPO_ROOT / "choi_covariance_cpu" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

alpha_values = [1.0, 1.5, 1.8, 1.9, 2.0, 2.1, 2.2, 2.5, 3.0]
ny_values = [16, 24, 32]
gap_nan_sentinel = 10.0

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["CMU Sans Serif", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print(f"campaign root: {CAMPAIGN_ROOT}")
print(f"NaN visualization sentinel: {gap_nan_sentinel}")


In [ ]:
if not (CAMPAIGN_ROOT / "campaign_manifest.json").exists():
    raise FileNotFoundError(f"Missing completed campaign manifest: {CAMPAIGN_ROOT / 'campaign_manifest.json'}")

manifest = json.loads((CAMPAIGN_ROOT / "campaign_manifest.json").read_text())
if manifest.get("status") != "completed":
    raise RuntimeError(f"Campaign is not complete: status={manifest.get('status')!r}")

rows = []
for run_dir in sorted((CAMPAIGN_ROOT / "runs").iterdir()):
    if not run_dir.is_dir():
        continue
    summary = json.loads((run_dir / "run_summary.json").read_text())
    ny = int(summary["Ny"])
    alpha_1 = float(summary["alpha_1"])
    if ny not in ny_values or alpha_1 not in alpha_values:
        continue
    with np.load(run_dir / "particle_choi_transfer_gap_vs_cycle.npz", allow_pickle=False) as data:
        cycles = np.asarray(data["cycles"], dtype=np.int64)
        gap = np.asarray(data["gap"], dtype=np.float64)
        finite_count = np.asarray(data["finite_exponent_count"], dtype=np.int64)
        final_gap = gap[:, -1]
        final_finite = final_gap[np.isfinite(final_gap)]
        final_gap_plot = np.where(np.isfinite(final_gap), final_gap, gap_nan_sentinel)
        first_zero_cycles = []
        for sample_counts in finite_count:
            zero_idx = np.where(sample_counts == 0)[0]
            first_zero_cycles.append(int(cycles[zero_idx[0]]) if zero_idx.size else -1)
    positive_zero = [cycle for cycle in first_zero_cycles if cycle > 0]
    rows.append({
        "Ny": ny,
        "alpha_1": alpha_1,
        "cycles_final": int(cycles[-1]),
        "finite_sample_count_final": int(final_finite.size),
        "finite_rms_gap_final": float(np.sqrt(np.mean(final_finite**2))) if final_finite.size else np.nan,
        "finite_mean_gap_final": float(np.mean(final_finite)) if final_finite.size else np.nan,
        "sentinel_rms_gap_final": float(np.sqrt(np.mean(final_gap_plot**2))),
        "median_first_zero_cycle": float(np.median(positive_zero)) if positive_zero else np.nan,
        "run_dir": str(run_dir),
    })

rows = sorted(rows, key=lambda row: (row["Ny"], row["alpha_1"]))
expected = len(ny_values) * len(alpha_values)
if len(rows) != expected:
    raise RuntimeError(f"Expected {expected} rows, loaded {len(rows)}")

print(f"loaded {len(rows)} completed config rows")
for row in rows:
    print(
        f"Ny={row['Ny']}, alpha_1={row['alpha_1']:g}: "
        f"finite_final={row['finite_sample_count_final']}/10, "
        f"finite_rms={row['finite_rms_gap_final']:.6g}, "
        f"plot_rms={row['sentinel_rms_gap_final']:.6g}, "
        f"median_zero_cycle={row['median_first_zero_cycle']}"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.6), constrained_layout=True)
colors = {16: "#1f77b4", 24: "#d62728", 32: "#2ca02c"}
markers = {16: "o", 24: "s", 32: "^"}

for ny in ny_values:
    subset = [row for row in rows if row["Ny"] == ny]
    alphas = np.asarray([row["alpha_1"] for row in subset], dtype=float)
    plotted = np.asarray([row["sentinel_rms_gap_final"] for row in subset], dtype=float)
    finite_only = np.asarray([row["finite_rms_gap_final"] for row in subset], dtype=float)
    ax.plot(alphas, plotted, color=colors[ny], marker=markers[ny], markersize=3.4, linewidth=1.25, label=fr"$N_y={ny}$")
    finite_mask = np.isfinite(finite_only)
    ax.scatter(alphas[finite_mask], finite_only[finite_mask], color=colors[ny], s=10, alpha=0.45, linewidth=0)

ax.axhline(gap_nan_sentinel, color="0.35", linestyle="--", linewidth=0.8, label="undefined sentinel")
ax.set_yscale("log")
ax.set_xlabel(r"$\alpha_1$")
ax.set_ylabel("final RMS gap")
ax.set_title(r"$N_x=16$, final cycle $C=2N_y$")
ax.set_xticks(alpha_values)
ax.tick_params(axis="x", rotation=35)
ax.grid(True, which="both", color="0.88", linewidth=0.5)
ax.legend(frameon=False, loc="best")

png_path = FIGURE_DIR / "nx16_alpha1_sweep_final_gap_vs_alpha.png"
pdf_path = FIGURE_DIR / "nx16_alpha1_sweep_final_gap_vs_alpha.pdf"
fig.savefig(png_path, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {png_path}")
print(f"saved {pdf_path}")
plt.show()


In [ ]:
header = "| Ny | alpha_1 | finite final samples | finite-only final RMS | plotted RMS | median zero-finite cycle |"
print(header)
print("|---:|---:|---:|---:|---:|---:|")
for row in rows:
    finite_rms = "undefined" if not np.isfinite(row["finite_rms_gap_final"]) else f"{row['finite_rms_gap_final']:.6g}"
    zero_cycle = "-" if not np.isfinite(row["median_first_zero_cycle"]) else f"{row['median_first_zero_cycle']:.1f}"
    print(
        f"| {row['Ny']} | {row['alpha_1']:g} | {row['finite_sample_count_final']}/10 | "
        f"{finite_rms} | {row['sentinel_rms_gap_final']:.6g} | {zero_cycle} |"
    )
